In [5]:
import os
import json
from kafka import KafkaConsumer

# Configurable via env vars for easy K8s integration
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka.apache-kafka.svc.cluster.local:9092")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC", "sliding_window_lstm")
KAFKA_GROUP_ID = os.getenv("KAFKA_GROUP_ID", "cpu-window-consumer")

def main():
    consumer = KafkaConsumer(
        KAFKA_TOPIC,
        bootstrap_servers=[s.strip() for s in KAFKA_BOOTSTRAP_SERVERS.split(",") if s.strip()],
        group_id=None, # resume if offset exists
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        auto_offset_reset="latest",  # start at end if no offset stored
        enable_auto_commit=False # save progress automatically
    )
    

    print(f"Connected to Kafka topic '{KAFKA_TOPIC}'. Waiting for windows...")
    for msg in consumer:
        window = msg.value
        # Print window in a clean, compact way: time value | time value | ...
        print(" | ".join(f"{item['ts'].split()[-1]} {item['cpu_pct']:.2f}" for item in window))

if __name__ == "__main__":
    main()


Connected to Kafka topic 'sliding_window_lstm'. Waiting for windows...
17:20:00 16.40 | 17:20:05 16.40 | 17:20:10 16.40 | 17:20:15 16.40 | 17:20:20 17.96


KeyboardInterrupt: 